#### This Jupyter Notebook will demonstrate how to perform an in silico digestion of a protein sequence using the `pyteomics` package.

### Step 1: Importing Required Libraries



In [8]:
from Bio import SeqIO
from pyteomics import parser
import plotly.graph_objects as go
from collections import defaultdict as ddict
import csv
import pandas
import sys
from deeplc import DeepLC

### Step 2: Digesting all proteins and predicting RTs


In [17]:
# Parsing fasta file
pep2prot = {}
for record in SeqIO.parse("maripaludis_proteome.fasta", "fasta"):
    protein_id = str(record.id)
    sequence = str(record.seq)
    # use pyteomics to generate tryptic peptides
    tryptic_peptides = parser.cleave(sequence, "trypsin")
    for pep in tryptic_peptides:
        if pep in pep2prot.keys():
            pep2prot[pep].append(protein_id)
        else:
            pep2prot[pep] = [protein_id]

# Generating dataframe for DeepLC
peptides = []
modifications = []
for pep in pep2prot.keys():
    if len(pep) >5 and len(pep) < 60:
        peptides.append(pep)
        modifications.append('')
df = pandas.DataFrame({'seq':peptides, 'modifications':modifications})

# Running DeepLC
dlc = DeepLC()
predictions = dlc.make_preds(seq_df=df, calibrate=False)

# Saving results as csv
df = pandas.DataFrame({'seq':peptides, 'modifications':modifications, 'RTs': predictions})
df.to_csv('maripaludis_proteome_rt_predictions.csv')

29/29 [==============================] - 17s 571ms/step


### Step 3: Parsing CSV file with predicted RTs


In [24]:
# Load the CSV directly into a DataFrame
df = pandas.read_csv('maripaludis_proteome_rt_predictions.csv')

# Perform the calculation on the entire column at once (Vectorization)
# This replaces the 'times' loop and the 'normalized_times' loop
normalized_times = (df["RTs"] / 10) * 60

# If you still need the peptides as a list for other logic:
peptides = df["seq"].tolist()

print("parsing done")

# Plotting
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=normalized_times, 
    marker=dict(color='rgb(0, 0, 0)') 
))

fig.update_xaxes(title_text="Retention Time (min)")
fig.update_yaxes(title_text="Number of Occurrences")
fig.show(renderer="iframe")

parsing done


#### Step 4: Calculating overlapping peaks

